# ICS 604: APPLIED DATA SCIENCE

## Hierarchical Indexes on DataFrames and Series

---


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
np.random.seed(12345)

## Hierarchical Indexing

Hierarchical indexing allows pandas objects to have **multiple index levels** along an axis. We briefly encountered this concept earlier when working with `groupby`, where grouping by more than one column naturally produces a hierarchical index.

Despite the term *indexing*, hierarchical indexing applies to both row indexes and column indexes, enabling more complex and structured data representations. In a `MultiIndex` object, each individual index is referred to as a **level**, and together these levels form a hierarchy that allows for more expressive data selection and organization.

In [ ]:
prog_languages = pd.DataFrame({"Course": ["Python", "Rust", "Python", "Rust"], 
                               "Nb_participants": [10, 30, 27, 18]})
prog_languages

In [ ]:
# group by a single column; access each group separately
x = prog_languages.groupby("Course")
p = x.get_group("Python")
r = x.get_group("Rust")

display(p, r)

In [ ]:
prog_languages.groupby("Course").sum()

In [ ]:
# MultiIndex created by pairing elements position-wise (zip-style)
data = pd.Series(np.random.randn(9),
                 index=[['a', 'a', 'a', 'b', 'b', 'c', 'c', 'd', 'd'],
                        [1, 2, 3, 1, 3, 1, 2, 2, 3]])
data

In [ ]:
data.index

In [ ]:
# access a single element - exact lookup of a single MultiIndex key
print(data[('b', 1)])

In [ ]:
print(data['b', 1])

In [ ]:
# selects all entries where the first index level is 'b'
data[('b',)]

In [ ]:
# label-based slice on the first level of the MultiIndex
data['b':'c']

In [ ]:
# label-based selection on level 0 for multiple keys
data.loc[['b', 'd']]

In [ ]:
# selection from an "inner" level
data.loc[:, 2]

In [ ]:
data[:, 2]

In [ ]:
df = pd.DataFrame({"A": np.random.randn(9), "B": np.random.randn(9)}, 
                  index=[['a', 'a', 'a', 'b', 'b', 'c', 'c', 'd', 'd'], [1, 2, 3, 1, 3, 1, 2, 2, 3]])
df

In [ ]:
# data.loc[:, 2] worked on a Series but 
# df.loc[:, 2] does not work on this DataFrame

df.loc[:, 2]

In [ ]:
# Boolean selection on the second MultiIndex level
df.loc[df.index.get_level_values(1) == 2]

In [ ]:
df.index.get_level_values(1) == 2

In [ ]:
# Row-wise slicing of a MultiIndex using IndexSlice
df.loc(axis=0)[pd.IndexSlice[:, 2]]

In [ ]:
df.loc[pd.IndexSlice[:, 2], :]

### Stacked vs. Unstacked Data

Hierarchical, or stacked, data often reflects how information is collected in practice. For example, in a hospital setting, different measurements for patients may be recorded in separate files or at different times. Each row represents a single observation, identified by a patient ID and a variable name (such as LDL, HDL, or VO2), along with its value. This naturally leads to a stacked representation, where multiple variables are recorded across rows rather than spread across columns.

```python
    File 1
    patient_ABC  LDL 112
    patient_ABC  HDL 48
    patient_CCX  LDL 112
    patient_VDM  LDL 112
    patient_ABC  VO2 112
    patient_CCZ  RER 48
    ...
```

This format is especially useful when not all variables are measured for every subject. If each variable were stored as a separate column, many entries would be missing, resulting in a wide table with a large number of empty cells. Using a stacked or hierarchical structure avoids this sparsity and provides a more flexible and efficient way to store and manage partially observed data.

### Stacking and Unstacking a DataFrame

Pandas allows you to convert between a regular DataFrame and a hierarchical (MultiIndex) representation using the `stack()` and `unstack()` methods. These operations reshape the data by moving one level of the columns into the index or vice versa.

The `stack()` method pivots columns into a new inner index level, producing a taller, stacked DataFrame or Series. Conversely, `unstack()` moves one level of the index back into columns, creating a wider, unstacked DataFrame. Together, these methods make it easy to switch between stacked and unstacked views of the same data, depending on which format is more convenient for analysis or visualization.

<img src="https://www.dropbox.com/scl/fi/bw3bz0dwgwlnufsqeuv4n/stacking_DF.png?rlkey=ipag93cjduudc9unfwijr2bth&st=swr1p0h2&dl=1">

In [ ]:
x = pd.DataFrame(np.random.randn(12).reshape((4, 3)), 
                 index=list("abcd"), columns=[1, 2, 3])
x.loc["b", 2] = np.nan
x.loc["c", 3] = np.nan
x.loc["d", 1] = np.nan
x

In [ ]:
x  = x.stack()
x

In [ ]:
x = x.unstack()
x

### Hierarchical Indexes on Rows and Columns

In a pandas DataFrame, both rows and columns can use hierarchical (MultiIndex) structures. This means you can organize data with multiple levels of labels along either axis, allowing for rich, multi-dimensional representations within a two-dimensional table.

It is important to remember that row labels are stored in the `index` attribute, while column labels are stored in the `columns` attribute. When either of these is a `MultiIndex`, each level represents a different layer of categorization.

<img src="https://www.dropbox.com/scl/fi/xzgrl00rcgcxoutusv2ya/index_columns.png?rlkey=d2c2ru42jjmqlovswe1kqqbja&st=8rfqqm74&dl=1" width=400>

In [ ]:
data = pd.DataFrame(np.arange(12).reshape((4, 3)),
                     index=[['a', 'a', 'b', 'b'], [1, 2, 1, 2]],
                     columns=[['Ohio', 'Ohio', 'Colorado'],
                              ['Green', 'Red', 'Green']])
data

In [ ]:
# name hierarchical levels

data.index.names = ['key1', 'key2']
data.columns.names = ['state', 'color']
data

In [ ]:
print(data.index.get_level_values("key1"))

In [ ]:
print(data.index.get_level_values(level=0))

In [ ]:
print(data.index.get_level_values("key2"))

In [ ]:
print(data.index.get_level_values(1))

In [ ]:
print(data.columns.get_level_values("state"))

In [ ]:
print(data.columns.get_level_values(0))

In [ ]:
print(data.columns.get_level_values("color"))

In [ ]:
print(data.columns.get_level_values(1))

#### Questions

- What type of variable would the following return (DataFrame, Series, np.array, ...)?

```python
    data['Ohio']`
```


- How about the following?

```python
    data['Ohio', 'Green']`
```

In [ ]:
# Partial selection; Level dropping

print(type(data['Ohio']))
data['Ohio']

In [ ]:
print(type(data['Ohio', 'Green']))

data['Ohio', 'Green']

In [ ]:
# Explicit slicing; Retain level

data.loc[:, pd.IndexSlice['Ohio', ['Red', 'Green']]]

In [ ]:
# When slicing to a single column → pandas collapses the column MultiIndex into a simple name 
data.loc[:, pd.IndexSlice['Ohio', 'Green']]

### Creating and Assigning a `MultiIndex` Object

Pandas provides several constructors for creating a MultiIndex, such as building one directly from arrays. These constructors allow you to explicitly define multiple levels of labels that together form a hierarchical index.

Once created, a `MultiIndex` can be assigned to either the `index` or the `columns` attribute of a DataFrame (or to the `index` of a Series), as long as the dimensions are compatible. This makes it possible to add hierarchical structure to existing data and reorganize it for more expressive analysis.

In [ ]:
data = pd.DataFrame(np.arange(12).reshape((4, 3)),
                     index=[['a', 'a', 'b', 'b'], [1, 2, 1, 2]])
data.index.names = ["letter", "number"]
data

In [ ]:
col_idx = pd.MultiIndex.from_arrays([['Ohio', 'Ohio', 'Colorado'], ['Green', 'Red', 'Green']],
                                names=['state', 'color'])
col_idx

In [ ]:
data.columns = col_idx
data

In [ ]:
row_idx = pd.MultiIndex.from_product([['a', 'b'], [1, 2]], names=['letter', 'number'])
row_idx            

In [ ]:
data = pd.DataFrame(np.arange(12).reshape((4, 3)), index=row_idx, columns=col_idx)
data

### Reordering and Sorting Levels

Pandas provides tools for reorganizing hierarchical indexes. The `swaplevel()` method allows you to interchange two levels of a `MultiIndex` by specifying their level numbers or names, returning a new object with the levels reordered.

Just like with simple indexes, `MultiIndex` objects can also be sorted. You can sort by a single level or by a combination of levels, making it easier to control the order of hierarchical data for analysis, display, or further processing.

In [ ]:
data = pd.DataFrame(np.arange(12).reshape((4, 3)),
                     index=[['a', 'a', 'b', 'b'], [1, 2, 1, 2]])
data.index.names = ['key1', 'key2']
data

In [ ]:
data.swaplevel('key1', 'key2')

In [ ]:
data.sort_index(level=1, ascending=False)

In [ ]:
data.sort_index(level="key2", ascending=False)

In [ ]:
# You can also pass the level's index number, instead of name
data.swaplevel(0, 1).sort_index(level=0, ascending=False)

In [ ]:
data.sort_index(level=1, ascending=False).swaplevel(0, 1)

In [ ]:
data.columns = pd.MultiIndex.from_arrays([['Ohio', 'Ohio', 'Colorado'], ['Green', 'Red', 'Green']],
                       names=['state', 'color'])
data

In [ ]:
# The 3rd parameter specifies the axis; by default, axis=0
data.swaplevel(0, 1, 0)

In [ ]:
data.swaplevel(0, 1, 1)

In [ ]:
data.swaplevel('state', 'color', axis=1)

In [ ]:
data.swaplevel('state', 'color', 1)

### Summary Statistics by Level

Summary statistics in pandas can also be computed by level when working with hierarchical (MultiIndex) data. This allows you to apply familiar operations such as sums, means, or counts to a specific level of the index.

When a function is applied at a given level, the remaining levels are effectively collapsed (or “squashed”), and the result is aggregated over those levels. Internally, this behavior is equivalent to grouping the data by the chosen level, which can be explicitly achieved using the `groupby()` method on the desired index level.

In [ ]:
data

In [ ]:
data.groupby(level='key1').sum()

In [ ]:
data.groupby(level='key2').sum()

In [ ]:
data.groupby(level="state", axis=1).sum()

In [ ]:
data.T.groupby(level='state').sum().T

In [ ]:
data.T.groupby(level='state').sum()

In [ ]:
data.T.groupby(level='color').sum().T

### Setting and Resetting the Index

It is common to load a DataFrame from a file and then use one or more of its columns as the index. In pandas, this is done using the `set_index()` method, which promotes one or more existing columns to become the DataFrame’s index. By default, the columns used to create the index are removed from the DataFrame, though you can choose to keep them if needed.

The `reset_index()` method performs the opposite operation. It moves the current index back into the DataFrame as one or more regular columns and replaces the index with a new default `RangeIndex` (from 0 to n−1). 

In [ ]:
data = pd.DataFrame({'a': range(7), 'b': range(7, 0, -1),
                     'c': ['one', 'one', 'one', 'two', 'two', 'two', 'two'],
                     'd': [0, 1, 2, 0, 1, 2, 3]})
data

In [ ]:
data2 = data.set_index('b')
data2

In [ ]:
data3 = data.set_index(['c', 'd'])
data3

In [ ]:
data.set_index(['c', 'd'], drop=False)

In [ ]:
display(data2)

data4 = data2.reset_index()
data4

In [ ]:
data4.index

In [ ]:
display(data3)
data3.reset_index((1,))

In [ ]:
data3.reset_index((1,), drop=True)

In [ ]:
display(data3)
data3.reset_index(level=['d'])